In [ ]:
import time
import os
import csv
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

URL = "https://iuh.edu.vn/vi/hop-tac.html/p={p}"
LABEL = "Hợp tác"
CSV_FILE = "hop_tac.csv"

# ✅ đợi đúng link bài viết (h3 a) có href .html
ARTICLE_LINK_CSS = "h3 a[href$='.html']"

def make_driver(headless=True):
    options = webdriver.ChromeOptions()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--window-size=1400,900")
    return webdriver.Chrome(options=options)

def scroll_bottom(driver, rounds=3, pause=0.6):
    for _ in range(rounds):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(pause)

driver = make_driver(headless=True)

file_exists = os.path.exists(CSV_FILE)
seen_href = set()  # ✅ chống trùng + ổn định hơn seen theo text

try:
    with open(CSV_FILE, mode="a", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f)

        if not file_exists:
            writer.writerow(["STT", "Thông tin", "Label"])
            stt = 1
        else:
            with open(CSV_FILE, encoding="utf-8-sig") as rf:
                stt = sum(1 for _ in rf)  # gồm header

        for p in range(0, 8):
            driver.get(URL.format(p=p))

            # ✅ đợi trang load xong
            WebDriverWait(driver, 25).until(
                lambda d: d.execute_script("return document.readyState") == "complete"
            )

            # ✅ đợi đúng link bài viết xuất hiện
            WebDriverWait(driver, 25).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, ARTICLE_LINK_CSS))
            )

            # ✅ scroll để tránh lazy-load miss
            scroll_bottom(driver, rounds=3, pause=0.5)

            links = driver.find_elements(By.CSS_SELECTOR, ARTICLE_LINK_CSS)

            added = 0
            for a in links:
                href = a.get_attribute("href") or ""
                if not href or href in seen_href:
                    continue

                text = a.text.strip()
                if not text:
                    continue

                seen_href.add(href)
                writer.writerow([stt, text, LABEL])
                f.flush()
                print(f"✔ p={p} wrote row {stt}")
                stt += 1
                added += 1

                time.sleep(1)

            print(f"✅ page {p} done | links={len(links)} | added={added}")
            time.sleep(1)

finally:
    driver.quit()

print("🎉 DONE – CSV được ghi từng dòng")


✔ p=0 wrote row 1
✔ p=0 wrote row 2
✔ p=0 wrote row 3
✔ p=0 wrote row 4
✔ p=0 wrote row 5
✔ p=0 wrote row 6
✔ p=0 wrote row 7
✔ p=0 wrote row 8
✔ p=0 wrote row 9
✔ p=0 wrote row 10
✔ p=0 wrote row 11
✔ p=0 wrote row 12
✔ p=0 wrote row 13
✔ p=0 wrote row 14
✔ p=0 wrote row 15
✅ page 0 done | links=15 | added=15
✔ p=1 wrote row 16
✔ p=1 wrote row 17
✔ p=1 wrote row 18
✔ p=1 wrote row 19
✔ p=1 wrote row 20
✔ p=1 wrote row 21
✔ p=1 wrote row 22
✔ p=1 wrote row 23
✔ p=1 wrote row 24
✔ p=1 wrote row 25
✔ p=1 wrote row 26
✔ p=1 wrote row 27
✔ p=1 wrote row 28
✔ p=1 wrote row 29
✔ p=1 wrote row 30
✅ page 1 done | links=15 | added=15
✔ p=2 wrote row 31
✔ p=2 wrote row 32
✔ p=2 wrote row 33
✔ p=2 wrote row 34
✔ p=2 wrote row 35
✔ p=2 wrote row 36
✔ p=2 wrote row 37
✔ p=2 wrote row 38
✔ p=2 wrote row 39
✔ p=2 wrote row 40
✔ p=2 wrote row 41
✔ p=2 wrote row 42
✔ p=2 wrote row 43
✔ p=2 wrote row 44
✔ p=2 wrote row 45
✅ page 2 done | links=15 | added=15
✔ p=3 wrote row 46
✔ p=3 wrote row 47
✔ p=3 wr

In [1]:
import pandas as pd
test_df = pd.read_csv("iuh-data-gioi-thieu-chung.csv")

In [2]:
test_df

,STT,Thông tin,Label
0,1,Tiền thân của Đại học Công nghiệp Thành phố Hồ...,giới thiệu
1,2,"Năm 1968, Trường được đổi tên thành Trường tư ...",giới thiệu
2,3,"Năm 1978, Trường được đổi tên thành Trường Côn...",giới thiệu
3,4,"Tháng 3 năm 1999, Trường được nâng cấp thành T...",giới thiệu
4,5,"Trụ sở chính tại Số 12 Nguyễn Văn Bảo, Phường ...",giới thiệu
...,...,...,...
148,149,Chức năng của Phòng Công tác chính trị và hỗ t...,NaN
149,150,Chức năng của Phòng Công tác chính trị và hỗ t...,NaN
150,151,Chức năng của Phòng Công tác chính trị và hỗ t...,NaN
151,152,Chức năng của Phòng Công tác chính trị và hỗ t...,NaN


In [ ]:
import requests
from bs4 import BeautifulSoup
import csv
import time

base_url = "https://iuh.edu.vn/vi/"

faculties = {
    "Công nghệ Cơ khí": "khoa-cong-nghe-co-khi",
    "Công nghệ Thông tin": "khoa-cong-nghe-thong-tin",
    "Công nghệ Điện": "khoa-cong-nghe-dien",
    "Công nghệ Điện tử": "khoa-cong-nghe-dien-tu",
    "Công nghệ Động lực": "khoa-cong-nghe-dong-luc",
    "Công nghệ Nhiệt - Lạnh": "khoa-cong-nghe-nhiet-lanh",
    "Công nghệ May - Thời trang": "khoa-cong-nghe-may-thoi-trang",
    "Công nghệ Hóa học": "khoa-cong-nghe-hoa-hoc",
    "Khoa học Cơ bản": "khoa-khoa-hoc-co-ban",
    "Luật & KH Chính trị": "khoa-luat-va-khoa-hoc-chinh-tri",
    "Ngoại ngữ": "khoa-ngoai-ngu",
    "Quản trị Kinh doanh": "khoa-quan-tri-kinh-doanh",
    "Thương mại - Du lịch": "khoa-thuong-mai-du-lich",
    "Kỹ thuật Xây dựng": "khoa-ky-thuat-xay-dung",
    "Khoa học Sức khỏe": "khoa-khoa-hoc-suc-khoe"
}

headers = {"User-Agent": "Mozilla/5.0"}

data = []

for name, slug in faculties.items():
    url = base_url + slug + ".html"
    print("Đang crawl:", url)

    try:
        res = requests.get(url, headers=headers, timeout=15)
        soup = BeautifulSoup(res.text, "html.parser")

        # Tiêu đề
        title = soup.find("h1").get_text(strip=True)

        # Nội dung chính
        article = soup.select_one(".iuhArticleContent")
        content = article.get_text(separator="\n", strip=True) if article else ""

        data.append([title, url, content])

        time.sleep(1)

    except Exception as e:
        print("Lỗi:", e)

# Lưu CSV
with open("iuh_faculties_full.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(["Faculty", "URL", "Content"])
    writer.writerows(data)

print("✅ Hoàn tất → iuh_faculties_full.csv")

MenuLãnh đạoĐảng ủyBan Giám hiệuPhòng banPhòng Tổ chức - Hành chínhPhòng Tài chính - Kế toánPhòng Kế hoạch - Đầu tưPhòng Đào tạoPhòng Quản lý khoa học và hợp tác quốc tếPhòng Công tác chính trị và hỗ trợ sinh viênPhòng Khảo thí và Đảm bảo chất lượngPhòng quản trịPhòng Dịch vụPhòng Quản lý Ký túc xáTạp chí Khoa học và Công nghệ - IUHNhà xuất bản Đại học Công nghiệp TP. Hồ Chí MinhKhoaKhoa Công nghệ Cơ khíKhoa Công nghệ thông tinKhoa Công nghệ ĐiệnKhoa Công nghệ Điện tửKhoa Công nghệ Động lựcKhoa Công nghệ Nhiệt - LạnhKhoa Công nghệ May - Thời trangKhoa Công nghệ Hóa họcKhoa Khoa học Cơ bảnKhoa Luật và Khoa học chính trịKhoa Ngoại ngữKhoa Quản trị Kinh doanhKhoa Thương mại - Du lịchKhoa Kỹ thuật Xây dựngKhoa Khoa học Sức khỏeViệnViện Đào tạo Quốc tế và Sau đại họcViện Tài chính - Kế toánViện Công nghệ Sinh học và Thực phẩmViện Khoa học Công nghệ và Quản lý Môi trườngTrung tâmTrung tâm Quản trị Hệ thốngTrung tâm Thông tin - Truyền thôngTrung tâm Ngoại ngữTrung tâm Giáo dục quốc phòng và T

✅ Hoàn tất → iuh_faculties_full.txt
